# Phase 5 Kaggle Experiment Runner

Run one explicit train/test Phase 5 experiment at a time. This notebook is prepared locally and should be executed later on Kaggle with a GPU.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import time

EXPERIMENT_NAME = "fixed_30"  # fixed_30, fixed_60, fixed_100, automatic_60, full
PROJECT_REPO_URL = "https://github.com/Kbharathreddy/CV.git"
PROJECT_DIR = Path("/kaggle/working/CV")
WORK_ROOT = Path("/kaggle/working")
SOURCE_PATH = Path("/kaggle/input/datasets/kopperlabharathreddy/mip-nerf-360-bonsai-our-colmap-reconstruction/bonsai")
SOURCE_IMAGES = SOURCE_PATH / "images"
SOURCE_MODEL = SOURCE_PATH / "sparse" / "0"
PHASE5_ROOT = WORK_ROOT / "outputs" / "phase5" / EXPERIMENT_NAME
MODEL_PATH = PHASE5_ROOT / "model"
TRAIN_SCENE = PHASE5_ROOT / "train_scene"
TEST_SCENE = PHASE5_ROOT / "test_scene"
GRAPHDECO_DIR = WORK_ROOT / "gaussian-splatting"
GRAPHDECO_REPO = "https://github.com/graphdeco-inria/gaussian-splatting.git"
GRAPHDECO_COMMIT = "54c035f7834b564019656c3e3fcc3646292f727d"
ITERATION_OVERRIDE = None  # set an integer to override the manifest, e.g. 7000
LPIPS_NETWORK = "alex"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("SOURCE_PATH:", SOURCE_PATH)
print("PHASE5_ROOT:", PHASE5_ROOT)

## 1. GPU Verification

In [ ]:
def run(cmd, *, cwd=None, check=True, capture_output=False):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=capture_output)

run(["nvidia-smi"])
run([sys.executable, "--version"])
run(["git", "--version"])
import torch
assert torch.cuda.is_available(), "CUDA is required for Phase 5 training"
GPU_NAME = torch.cuda.get_device_name(0)
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("gpu:", GPU_NAME)

## 2. Locate Bonsai Input

In [ ]:
required_sparse = ["cameras.bin", "images.bin", "points3D.bin"]
assert SOURCE_IMAGES.is_dir(), f"Missing images directory: {SOURCE_IMAGES}"
assert SOURCE_MODEL.is_dir(), f"Missing sparse model directory: {SOURCE_MODEL}"
for name in required_sparse:
    assert (SOURCE_MODEL / name).exists(), f"Missing COLMAP file: {name}"
image_files = sorted([p for p in SOURCE_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
assert len(image_files) == 292, f"Expected 292 bonsai images, found {len(image_files)}"
print("image_count:", len(image_files))
print("first image:", image_files[0].name)
print("last image:", image_files[-1].name)

## 3. Clone Project and Pinned 3DGS

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_DIR)])
run(["git", "checkout", "master"], cwd=PROJECT_DIR)

if GRAPHDECO_DIR.exists():
    shutil.rmtree(GRAPHDECO_DIR)
run(["git", "clone", "--recursive", GRAPHDECO_REPO, str(GRAPHDECO_DIR)])
run(["git", "checkout", GRAPHDECO_COMMIT], cwd=GRAPHDECO_DIR)
run(["git", "submodule", "update", "--init", "--recursive"], cwd=GRAPHDECO_DIR)
actual_commit = run(["git", "rev-parse", "HEAD"], cwd=GRAPHDECO_DIR, capture_output=True).stdout.strip()
assert actual_commit == GRAPHDECO_COMMIT, actual_commit
print("Graphdeco commit:", actual_commit)

## 4. Install CUDA Dependencies

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "plyfile", "tqdm", "ninja", "scikit-image", "lpips"])
run([sys.executable, "-m", "pip", "install", "--quiet", "--no-build-isolation", "submodules/diff-gaussian-rasterization"], cwd=GRAPHDECO_DIR)
run([sys.executable, "-m", "pip", "install", "--quiet", "--no-build-isolation", "submodules/simple-knn"], cwd=GRAPHDECO_DIR)
import diff_gaussian_rasterization
import simple_knn
print("CUDA extensions import successfully")

## 5. Prepare Selected Experiment Scene

In [ ]:
manifest_path = PROJECT_DIR / "experiments" / "phase5" / f"{EXPERIMENT_NAME}.json"
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text())
ITERATIONS = int(ITERATION_OVERRIDE or manifest["configuration"]["iterations"])
assert ITERATIONS != 300, "Phase 5 full experiments must not accidentally use the Phase 4 smoke-test iteration count"

if PHASE5_ROOT.exists():
    shutil.rmtree(PHASE5_ROOT)
prep_cmd = [
    sys.executable,
    str(PROJECT_DIR / "scripts" / "prepare_phase5_experiment.py"),
    "--manifest-dir", str(PROJECT_DIR / "experiments" / "phase5"),
    "--source-images", str(SOURCE_IMAGES),
    "--source-model", str(SOURCE_MODEL),
    "--experiment", EXPERIMENT_NAME,
    "--output", str(PHASE5_ROOT),
    "--link-mode", "copy",
    "--overwrite",
]
run(prep_cmd, cwd=PROJECT_DIR)
print("train_scene:", TRAIN_SCENE)
print("test_scene:", TEST_SCENE)

## 6. Validate Explicit Train/Test Split

In [ ]:
train_images = set(manifest["training_images"])
test_images = set(manifest["test_images"])
assert train_images.isdisjoint(test_images), "Train/test overlap is forbidden"
assert len(train_images) == manifest["train_image_count"]
assert len(test_images) == manifest["test_image_count"]
assert len(list((TRAIN_SCENE / "images").iterdir())) == manifest["train_image_count"]
assert len(list((TEST_SCENE / "images").iterdir())) == manifest["test_image_count"]
for scene in (TRAIN_SCENE, TEST_SCENE):
    for name in required_sparse:
        assert (scene / "sparse" / "0" / name).exists(), f"Missing {name} in {scene}"
print("train images:", len(train_images))
print("held-out test images:", len(test_images))
print("held-out sample:", sorted(test_images)[:5])

## 7. Run 3DGS Training

In [ ]:
train_cmd = [
    sys.executable, "train.py",
    "-s", str(TRAIN_SCENE),
    "-m", str(MODEL_PATH),
    "-i", "images",
    "--iterations", str(ITERATIONS),
    "--test_iterations", "-1",
    "--save_iterations", str(ITERATIONS),
    "--checkpoint_iterations", str(ITERATIONS),
    "--disable_viewer",
]
start = time.perf_counter()
training = run(train_cmd, cwd=GRAPHDECO_DIR, capture_output=True)
training_runtime_seconds = time.perf_counter() - start
train_log = training.stdout + "\n" + training.stderr
(PHASE5_ROOT / "logs").mkdir(parents=True, exist_ok=True)
(PHASE5_ROOT / "logs" / "train.log").write_text(train_log)
loss_values = [float(match.group(1)) for match in re.finditer(r"(?<!Depth )Loss=([0-9eE+\-.]+)", train_log)]
assert len(loss_values) >= 2, "Expected at least two main Loss values"
assert all(value == value and value != float("inf") for value in loss_values)
initial_loss = loss_values[0]
final_loss = loss_values[-1]
checkpoint_path = MODEL_PATH / f"chkpnt{ITERATIONS}.pth"
point_cloud_path = MODEL_PATH / "point_cloud" / f"iteration_{ITERATIONS}" / "point_cloud.ply"
assert checkpoint_path.exists(), checkpoint_path
assert point_cloud_path.exists(), point_cloud_path
print("runtime:", training_runtime_seconds)
print("initial_loss:", initial_loss)
print("final_loss:", final_loss)

## 8. Render Held-Out Test Views

In [ ]:
render_cmd = [
    sys.executable, "render.py",
    "-s", str(TEST_SCENE),
    "-m", str(MODEL_PATH),
    "-i", "images",
    "--iteration", str(ITERATIONS),
    "--skip_test",
]
start = time.perf_counter()
rendering = run(render_cmd, cwd=GRAPHDECO_DIR, capture_output=True)
render_runtime_seconds = time.perf_counter() - start
(PHASE5_ROOT / "logs" / "render.log").write_text(rendering.stdout + "\n" + rendering.stderr)
render_dir = MODEL_PATH / "train" / f"ours_{ITERATIONS}" / "renders"
assert render_dir.is_dir(), render_dir
renders = sorted(render_dir.glob("*.png"))
assert len(renders) == manifest["test_image_count"], f"Expected held-out renders only, found {len(renders)}"
print("render_runtime_seconds:", render_runtime_seconds)
print("render_count:", len(renders))
print("example_render:", renders[0])

## 9. Evaluate Held-Out Metrics

In [ ]:
metrics_path = PHASE5_ROOT / "metrics.json"
per_image_csv = PHASE5_ROOT / "metrics_per_image.csv"
eval_cmd = [
    sys.executable,
    str(PROJECT_DIR / "scripts" / "evaluate_phase5.py"),
    "--ground-truth-dir", str(TEST_SCENE / "images"),
    "--render-dir", str(render_dir),
    "--manifest", str(manifest_path),
    "--output-json", str(metrics_path),
    "--per-image-csv", str(per_image_csv),
    "--lpips",
    "--lpips-network", LPIPS_NETWORK,
    "--lpips-device", "cuda",
]
run(eval_cmd, cwd=PROJECT_DIR)
metrics = json.loads(metrics_path.read_text())
print(json.dumps({k: metrics[k] for k in ["experiment", "test_image_count", "psnr_mean", "ssim_mean", "lpips_mean"]}, indent=2))

## 10. Save Standard Summary

In [ ]:
def count_ply_vertices(path):
    for line in path.read_text(errors="ignore").splitlines():
        if line.startswith("element vertex "):
            return int(line.split()[-1])
        if line.strip() == "end_header":
            break
    return None

summary = {
    "experiment": EXPERIMENT_NAME,
    "selection_method": manifest["selection_method"],
    "training_budget": manifest.get("training_budget"),
    "train_image_count": manifest["train_image_count"],
    "test_image_count": manifest["test_image_count"],
    "iterations": ITERATIONS,
    "gpu": GPU_NAME,
    "training_runtime_seconds": training_runtime_seconds,
    "render_runtime_seconds": render_runtime_seconds,
    "initial_loss": initial_loss,
    "final_loss": final_loss,
    "number_of_gaussians": count_ply_vertices(point_cloud_path),
    "psnr": metrics["psnr_mean"],
    "ssim": metrics["ssim_mean"],
    "lpips": metrics["lpips_mean"],
    "checkpoint_path": str(checkpoint_path),
    "point_cloud_path": str(point_cloud_path),
    "render_path": str(renders[0]),
    "status": "success",
    "warnings": ["Render directory name is Graphdeco 'train' because held-out views are supplied as a render-only scene."],
}
summary_path = PHASE5_ROOT / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, allow_nan=True))
print(summary_path)
print(json.dumps(summary, indent=2, allow_nan=True))

## 11. Archive Outputs

In [ ]:
archive_base = WORK_ROOT / f"phase5_{EXPERIMENT_NAME}"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=PHASE5_ROOT)
print("archive:", archive_path)